### Overview

The script automates the survey experiemnt and randomization using OpenAI's API by interacting with multiple virtual assistants. The assistnat should be created manually using OpenAI platform. It systematically poses questions loaded from Excel files to these assistants under varying treatment conditions. Each assistant's responses are recorded and then saved in an Excel file specific to that assistant.

### API KEY

For security, the script utilizes an `.env` file to store sensitive information such as the API key. 


## Check the impact of knowledge domain. Using Retrieval Augmented Generation (RAG)

This script automates a survey experiment to evaluate how different virtual assistants respond to questions about inflation expectations under various treatment conditions. 

In [ ]:
import os
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

# Load API key
load_dotenv('api.env')
api_key = os.getenv('api_key')
client = OpenAI(api_key=api_key)

# Questions directly in code
questions = {
    'Q1_I': "Over the last 12 months, what do you think the overall rate of inflation/deflation has been in the economy? The rate of inflation/deflation was [Your answer as a number] percent over the last 12 months.\nIf you think there was inflation, please enter a positive number. If you think there was deflation, please enter a negative number. If you think there was neither inflation nor deflation, please enter zero.",
    
    'Q2_I': "In THIS question, you will be asked about the probability (PERCENT CHANCE) of something happening. The percent chance must be a number between 0 and 100 and the sum of your answers must add up to 100.\nWhat do you think is the percent chance that, over the next 12 months...\n[RANGE OF EACH OPTION BELOW is 0-100 and each option can be 2 DECIMAL POINTS but the most important thing is that the total should be 100%]\n• the rate of deflation (opposite of inflation) will be 12% or more ______\n• the rate of deflation (opposite of inflation) will be between 8% and 12% ______\n• the rate of deflation (opposite of inflation) will be between 4% and 8% ______\n• the rate of deflation (opposite of inflation) will be between 2% and 4% ______\n• the rate of deflation (opposite of inflation) will be between 0% and 2% ______\n• the rate of inflation will be between 0% and 2% ______\n• the rate of inflation will be between 2% and 4% ______\n• the rate of inflation will be between 4% and 8% ______\n• the rate of inflation will be between 8% and 12% ______\n• the rate of inflation will be 12% or more______\nGive your answer as a list like this: [______%,______%,______%,______%,______%,______%,______%,______%,______%,______%]",
    
    'Q3_I': "In THIS question, you will be asked about the probability (PERCENT CHANCE) of something happening. The percent chance must be a number between 0 and 100 and the sum of your answers must add up to 100.\nWhat do you think is the percent chance that, 12-month period between April 2025 (24 months from now) and April 2026 (36 months from now)...\n[RANGE OF EACH OPTION BELOW is 0-100 and each option can be 2 DECIMAL POINTS but the most important thing is that the total should be 100%]\n• the rate of deflation (opposite of inflation) will be 12% or more ______\n• the rate of deflation (opposite of inflation) will be between 8% and 12% ______\n• the rate of deflation (opposite of inflation) will be between 4% and 8% ______\n• the rate of deflation (opposite of inflation) will be between 2% and 4% ______\n• the rate of deflation (opposite of inflation) will be between 0% and 2% ______\n• the rate of inflation will be between 0% and 2% ______\n• the rate of inflation will be between 2% and 4% ______\n• the rate of inflation will be between 4% and 8% ______\n• the rate of inflation will be between 8% and 12% ______\n• the rate of inflation will be 12% or more______\nGive your answer as a list like this: [______%,______%,______%,______%,______%,______%,______%,______%,______%,______%]",
    
    'Q1_F': "What do you expect the rate of inflation to be over the next 12 months? Please give your best guess. Over the next 12 months, I expect the rate of inflation to be ___ %",
    
    'Q2_F': "What do you expect the rate of inflation to be over that period? Please give your best guess. Over the 12-month period between April 2025 (24 months from now) and April 2026 (36 months from now), I expect the rate of inflation to be ___ %"
}

# Treatments directly in code
treatments = {
    'T_1': ("Control with no information", ""),
    'T_2': ("Placebo group", "Population of the U.S. grew by 1.2% between 2020 and 2022."),
    'T_3': ("Current rate, FFR", "The interest rate set by the Federal Reserve, known as the Federal Funds Rate, is currently at 5.25%."),
    'T_4': ("Inflation treatment: current inflation", "Over the last twelve months, the overall inflation rate in the economy as measured by the percentage change in a consumer price index has been 3.2%."),
    'T_5': ("Inflation treatment: current inflation + longer run", "Over the last twelve months, the overall inflation rate in the economy as measured by the percentage change in a consumer price index has been 3.2%. One forecast at the Federal Reserve is that this inflation rate will be 2.8% on average over the next few years and in the longer-run."),
    'T_6': ("Current fixed-rate 30-year mortgage", "The current average rate for fixed-rate 30-year mortgage is 7.5% per year.")
}

# Run number
Run = 1

# Assistant IDs
assistants = {
    f"Assistant_N_Run{Run}": "asst_lgNY6PdfBHbqs9PWHRDfdpNS",
    f"Assistant_W_Run{Run}": "asst_14e8DlzbaQAyhhT2qhh2LfRk",
    f"Assistant_E_Run{Run}": "asst_e9GUy0KOGCk2jjs5ldfNv56B",
    f"Assistant_M0_Run{Run}": "asst_v6zfG4XUS1LHX3FGF1WFCCdF",
    f"Assistant_M1_Run{Run}": "asst_w3CQEsx6N7Tgh09K0yk9sK1J",
    f"Assistant_M2_Run{Run}": "asst_YzMc4WAzTG6sIo9vSZPgwAVz"
}

# Number of runs per treatment
group_runs = {
    'T_1': 10, # Control Group
    'T_2': 10,
    'T_3': 10,
    'T_4': 10,
    'T_5': 10,
    'T_6': 10
}

# Common instructions for assistants
instructions = "Do not use the exact inflation if mentioned in the document. Use your general understanding of the document including the sentiments of the policy and all the information around it to answer. These are questions about inflation expectations and the perception of inflation, not inflation prediction. Do not answer nothing. You must ensures responses are solely numerical and formatted accordingly: for point estimates, it uses [______%], and for distribution estimates, it uses [______%,______%,______%,______%,______%,______%,______%,______%,______%,______%]. Under no circumestances do not any words and alphabets."

# For each assistant
for assistant_name, assistant_id in assistants.items():
    results = []
    
    # For each treatment group
    for treatment_group, num_runs in group_runs.items():
        treatment_title, treatment_info = treatments[treatment_group]
        
        # For each run
        for run in range(num_runs):
            result_row = [run, treatment_group]
            
            # Create a thread for this run
            my_thread = client.beta.threads.create()
            
            # Ask initial questions
            for question_id in [q for q in questions.keys() if q.endswith('_I')]:
                question = questions[question_id]
                
                # Add message with question
                client.beta.threads.messages.create(
                    thread_id=my_thread.id,
                    role="user",
                    content=question
                )
                
                # Run assistant
                my_run = client.beta.threads.runs.create(
                    thread_id=my_thread.id,
                    assistant_id=assistant_id,
                    instructions=instructions
                )
                
                # Wait for completion
                while True:
                    run_status = client.beta.threads.runs.retrieve(
                        thread_id=my_thread.id,
                        run_id=my_run.id
                    )
                    if run_status.status == "completed":
                        messages = client.beta.threads.messages.list(
                            thread_id=my_thread.id
                        )
                        response = messages.data[0].content[0].text.value
                        result_row.append(response)
                        break
            
            # Ask follow-up questions with treatment info
            for question_id in [q for q in questions.keys() if q.endswith('_F')]:
                question = questions[question_id]
                
                # Skip for control group (just add empty responses)
                if treatment_group == 'T_1':
                    result_row.append('')
                    continue
                
                # Add message with treatment info and follow-up question
                client.beta.threads.messages.create(
                    thread_id=my_thread.id,
                    role="user",
                    content=f"{treatment_info}\n\n{question}"
                )
                
                # Run assistant
                my_run = client.beta.threads.runs.create(
                    thread_id=my_thread.id,
                    assistant_id=assistant_id,
                    instructions=instructions
                )
                
                # Wait for completion
                while True:
                    run_status = client.beta.threads.runs.retrieve(
                        thread_id=my_thread.id,
                        run_id=my_run.id
                    )
                    if run_status.status == "completed":
                        messages = client.beta.threads.messages.list(
                            thread_id=my_thread.id
                        )
                        response = messages.data[0].content[0].text.value
                        result_row.append(response)
                        break
            
            # Add results and print progress
            results.append(result_row)
            print(f"{assistant_name}, {treatment_group}, Run {run + 1}: {result_row}")
    
    # Create and save results
    columns = ['Run', 'Group'] + [q for q in questions.keys() if q.endswith('_I')] + [q for q in questions.keys() if q.endswith('_F')]
    df = pd.DataFrame(results, columns=columns)
    df.to_excel(f'results_{assistant_name}.xlsx', index=False)
    print(f"Saved: results_{assistant_name}.xlsx")

# Pilot run with only GPT4o (with and without Persona)

In [ ]:
import os
import pandas as pd
import random
import time
from openai import OpenAI
from dotenv import load_dotenv

# Load API key
load_dotenv('api.env')
api_key = os.getenv('api_key')
client = OpenAI(api_key=api_key)

# Questions directly in code
questions = {
    'Q1_I': "Over the last 12 months, what do you think the overall rate of inflation/deflation has been in the economy? The rate of inflation/deflation was [Your answer as a number] percent over the last 12 months.\nIf you think there was inflation, please enter a positive number. If you think there was deflation, please enter a negative number. If you think there was neither inflation nor deflation, please enter zero.",
    
    'Q2_I': "In THIS question, you will be asked about the probability (PERCENT CHANCE) of something happening. The percent chance must be a number between 0 and 100 and the sum of your answers must add up to 100.\nWhat do you think is the percent chance that, over the next 12 months...\n[RANGE OF EACH OPTION BELOW is 0-100 and each option can be 2 DECIMAL POINTS but the most important thing is that the total should be 100%]\n• the rate of deflation (opposite of inflation) will be 12% or more ______\n• the rate of deflation (opposite of inflation) will be between 8% and 12% ______\n• the rate of deflation (opposite of inflation) will be between 4% and 8% ______\n• the rate of deflation (opposite of inflation) will be between 2% and 4% ______\n• the rate of deflation (opposite of inflation) will be between 0% and 2% ______\n• the rate of inflation will be between 0% and 2% ______\n• the rate of inflation will be between 2% and 4% ______\n• the rate of inflation will be between 4% and 8% ______\n• the rate of inflation will be between 8% and 12% ______\n• the rate of inflation will be 12% or more______\nGive your answer as a list like this: [______%,______%,______%,______%,______%,______%,______%,______%,______%,______%]",
    
    'Q3_I': "In THIS question, you will be asked about the probability (PERCENT CHANCE) of something happening. The percent chance must be a number between 0 and 100 and the sum of your answers must add up to 100.\nWhat do you think is the percent chance that, 12-month period between April 2025 (24 months from now) and April 2026 (36 months from now)...\n[RANGE OF EACH OPTION BELOW is 0-100 and each option can be 2 DECIMAL POINTS but the most important thing is that the total should be 100%]\n• the rate of deflation (opposite of inflation) will be 12% or more ______\n• the rate of deflation (opposite of inflation) will be between 8% and 12% ______\n• the rate of deflation (opposite of inflation) will be between 4% and 8% ______\n• the rate of deflation (opposite of inflation) will be between 2% and 4% ______\n• the rate of deflation (opposite of inflation) will be between 0% and 2% ______\n• the rate of inflation will be between 0% and 2% ______\n• the rate of inflation will be between 2% and 4% ______\n• the rate of inflation will be between 4% and 8% ______\n• the rate of inflation will be between 8% and 12% ______\n• the rate of inflation will be 12% or more______\nGive your answer as a list like this: [______%,______%,______%,______%,______%,______%,______%,______%,______%,______%]",
    
    'Q1_F': "What do you expect the rate of inflation to be over the next 12 months? Please give your best guess. Over the next 12 months, I expect the rate of inflation to be ___ %",
    
    'Q2_F': "What do you expect the rate of inflation to be over that period? Please give your best guess. Over the 12-month period between April 2025 (24 months from now) and April 2026 (36 months from now), I expect the rate of inflation to be ___ %",
    
    'Q3_F': "Can you explain your reasoning behind your inflation expectations? What factors are influencing your views?"
}

# Treatments directly in code
treatments = {
    'T_1': ("Control with no information", ""),
    'T_2': ("Placebo group", "Population of the U.S. grew by 1.2% between 2020 and 2022."),
    'T_3': ("Current rate, FFR", "The interest rate set by the Federal Reserve, known as the Federal Funds Rate, is currently at 5.25%."),
    'T_4': ("Inflation treatment: current inflation", "Over the last twelve months, the overall inflation rate in the economy as measured by the percentage change in a consumer price index has been 3.2%."),
    'T_5': ("Inflation treatment: current inflation + longer run", "Over the last twelve months, the overall inflation rate in the economy as measured by the percentage change in a consumer price index has been 3.2%. One forecast at the Federal Reserve is that this inflation rate will be 2.8% on average over the next few years and in the longer-run."),
    'T_6': ("Current fixed-rate 30-year mortgage", "The current average rate for fixed-rate 30-year mortgage is 7.5% per year.")
}

# Load personas from file
persona_df = pd.read_csv('persona.csv')

# Check for used personas file and load/update it
used_personas_file = 'used_personas.csv'
if os.path.exists(used_personas_file):
    used_personas = pd.read_csv(used_personas_file)
    persona_df = persona_df[~persona_df.index.isin(used_personas.index)]
else:
    used_personas = pd.DataFrame()

# Select personas for this run
num_personas = 600
selected_personas = persona_df.sample(n=num_personas, random_state=42)
used_personas = pd.concat([used_personas, selected_personas], ignore_index=True)
used_personas.to_csv(used_personas_file, index=False)

# Run number and assistants
Run = 1
assistants = {
    f"Assistant_NN_Run{Run}": "asst_pAz9kRNx09jYxnnc3OKS0gG2"
}

# Group allocation
group_runs = {
    'T_1': 1, 'T_2': 1, 'T_3': 1, 'T_4': 1, 'T_5': 1, 'T_6': 1
}
selected_personas['Group'] = pd.qcut(selected_personas.index, q=len(group_runs), labels=list(group_runs.keys()), duplicates='drop')

# Instructions
persona_instructions = "You are a {Age} year old {Gender} who is {Marital} with an education level of {Education} degree and income category of {Income} who lives in {STATE}. "
general_instructions = "Do not use the exact inflation if mentioned in the document. Use your general understanding of the document including the sentiments of the policy and all the information around it to answer. These are questions about inflation expectations and the perception of inflation, not inflation prediction. Do not answer nothing. You must ensure responses are solely numerical and formatted accordingly: for point estimates, it uses [______%], and for distribution estimates, it uses [______%,______%,______%,______%,______%,______%,______%,______%,______%,______%]. Under no circumstances do not use any words and alphabets."
q3f_instructions = "For this question ignore the prior instructions. Please provide a brief explanation for your answers in 1-2 sentences about why your answers."

# Function to get responses from OpenAI
def get_response(thread_id, assistant_id, content, instructions):
    client.beta.threads.messages.create(
        thread_id=thread_id,
        role="user",
        content=content
    )
    
    my_run = client.beta.threads.runs.create(
        thread_id=thread_id,
        assistant_id=assistant_id,
        instructions=instructions
    )
    
    while True:
        run_status = client.beta.threads.runs.retrieve(
            thread_id=thread_id,
            run_id=my_run.id
        )
        if run_status.status == "completed":
            messages = client.beta.threads.messages.list(
                thread_id=thread_id
            )
            return messages.data[0].content[0].text.value

# Function to process a batch of personas
def process_batch(batch, assistant_name, assistant_id, persona_type):
    results = []
    for _, persona in batch.iterrows():
        treatment_group = persona['Group']
        
        # Initialize the result row
        if persona_type == "with_persona":
            result_row = [0, treatment_group, persona['Age'], persona['Gender'], persona['Education'], persona['Marital'], persona['Income'], persona['STATE'], persona['userid']]
        else:
            result_row = [0, treatment_group] + [''] * 7  # Empty cells for persona info
        
        # Create a Thread
        my_thread = client.beta.threads.create()
        
        # Set instructions based on persona type
        if persona_type == "with_persona":
            instructions = persona_instructions.format(**persona.to_dict()) + general_instructions
        else:
            instructions = general_instructions
        
        # Initial questions (Q1_I, Q2_I, Q3_I)
        initial_responses = []
        initial_prompts = []
        for q in ['Q1_I', 'Q2_I', 'Q3_I']:
            question = questions[q]
            response = get_response(my_thread.id, assistant_id, question, instructions)
            initial_responses.append(response)
            initial_prompts.append(question)
        
        # Follow-up questions (Q1_F, Q2_F, Q3_F)
        followup_responses = []
        followup_prompts = []
        for q in ['Q1_F', 'Q2_F', 'Q3_F']:
            question = questions[q]
            # Only add treatment info for non-control groups
            if treatment_group != 'T_1':
                content = f"{treatments[treatment_group][1]}\n\n{question}"
            else:
                content = question
            
            # Use different instructions for Q3_F
            if q == 'Q3_F':
                response = get_response(my_thread.id, assistant_id, content, q3f_instructions)
            else:
                response = get_response(my_thread.id, assistant_id, content, instructions)
            
            followup_responses.append(response)
            followup_prompts.append(content)
        
        # Combine all responses and prompts
        all_responses = initial_responses + followup_responses
        all_prompts = initial_prompts + followup_prompts
        
        # Add responses, prompts, instructions, and treatment_info to the result row
        result_row.extend(all_responses)
        result_row.extend(all_prompts)
        result_row.append(instructions)
        result_row.append(treatments[treatment_group][1])
        
        results.append(result_row)
        
        print(f"Processed: {assistant_name}, {treatment_group}, {persona_type}")
    
    return results

# Main execution loop
for assistant_name, assistant_id in assistants.items():
    results_with_persona = []
    results_without_persona = []
    
    # Process both with and without persona information
    for persona_type in ["with_persona", "without_persona"]:
        personas = selected_personas if persona_type == "with_persona" else selected_personas.copy()
        personas = personas.sample(frac=1).reset_index(drop=True)  # Shuffle
        
        # Process in batches of 60
        for i in range(0, len(personas), 60):
            batch = personas.iloc[i:i+60]
            
            print(f"Batch {i//60 + 1} for {assistant_name} ({persona_type})")
            
            # Process the batch
            batch_results = process_batch(batch, assistant_name, assistant_id, persona_type)
            
            # Add to appropriate results list
            if persona_type == "with_persona":
                results_with_persona.extend(batch_results)
            else:
                results_without_persona.extend(batch_results)
            
            # Save intermediate results
            columns = ['Run', 'Group', 'Age', 'Gender', 'Education', 'Marital', 'Income', 'STATE', 'userid'] + \
                      ['Q1_I', 'Q2_I', 'Q3_I', 'Q1_F', 'Q2_F', 'Q3_F'] + \
                      ['Prompt_Q1_I', 'Prompt_Q2_I', 'Prompt_Q3_I', 'Prompt_Q1_F', 'Prompt_Q2_F', 'Prompt_Q3_F'] + \
                      ['Instructions', 'Treatment_Info']
            
            df_intermediate = pd.DataFrame(
                results_with_persona if persona_type == "with_persona" else results_without_persona, 
                columns=columns
            )
            df_intermediate.to_excel(f'intermediate_results_{assistant_name}_{persona_type}_batch_{i//60 + 1}.xlsx', index=False)
            
            # Wait between batches
            if i + 60 < len(personas):
                print("Waiting 5 minutes before next batch...")
                time.sleep(300)
    
    # Save final results
    columns = ['Run', 'Group', 'Age', 'Gender', 'Education', 'Marital', 'Income', 'STATE', 'userid'] + \
              ['Q1_I', 'Q2_I', 'Q3_I', 'Q1_F', 'Q2_F', 'Q3_F'] + \
              ['Prompt_Q1_I', 'Prompt_Q2_I', 'Prompt_Q3_I', 'Prompt_Q1_F', 'Prompt_Q2_F', 'Prompt_Q3_F'] + \
              ['Instructions', 'Treatment_Info']
    
    df_with_persona = pd.DataFrame(results_with_persona, columns=columns)
    df_without_persona = pd.DataFrame(results_without_persona, columns=columns)
    
    df_with_persona.to_excel(f'results_{assistant_name}_with_persona.xlsx', index=False)
    df_without_persona.to_excel(f'results_{assistant_name}_without_persona.xlsx', index=False)
    print(f"Results saved for {assistant_name}")